# NB0 · Tiếp nhận & kiểm tra toàn vẹn dữ liệu (Bước 0)
**FinReport Intelligence · Festival Chuyển đổi số 2026 · Bảng C · Case 3**

| | |
|---|---|
| **Đọc từ** | `01_datapack/CASE3_FINREPORT_INTELLIGENCE/` (chỉ đọc) |
| **Ghi ra** | `03_outputs/step0/`: `master_panel.csv`, `field_lineage.csv`, `integrity_checks.csv`, `integrity_report.md` |
| **Nhật ký** | `04_logs/audit_log.jsonl` (ghi thêm) |
| **Thư viện** | `lib/finreport_common.py` |

Chạy lần lượt từ trên xuống (hoặc **Runtime → Run all**).

## 1. Kết nối Drive & nạp thư viện dùng chung

In [ ]:
import os, sys
try:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT = "/content/drive/MyDrive/FinReport_Intelligence"
    IN_COLAB = True
except ImportError:                       # chạy ngoài Colab
    PROJECT = os.environ.get("FINREPORT_PROJECT", "FinReport_Intelligence")
    IN_COLAB = False

sys.path.insert(0, PROJECT + "/lib")
from finreport_common import *
cfg = setup(PROJECT)

## 2. (Chỉ lần đầu) Đưa Data Pack vào `01_datapack/`
Nếu ô trên báo `DATA_DIR ... (CHƯA CÓ)`, chạy ô này và chọn file zip Data Pack. Data Pack đã có thì ô này **không làm gì** (không ghi đè dữ liệu gốc).

In [ ]:
import zipfile, glob, shutil, tempfile
from pathlib import Path

if cfg.data_dir.exists():
    print("Data Pack đã có – bỏ qua:", cfg.data_dir)
elif IN_COLAB:
    from google.colab import files
    up = files.upload()
    with tempfile.TemporaryDirectory() as tmp:
        for name, content in up.items():
            zp = Path(tmp) / name
            zp.write_bytes(content)
            zipfile.ZipFile(zp).extractall(tmp)
        hit = glob.glob(f"{tmp}/**/02_metadata/dataset_manifest.csv", recursive=True)
        assert hit, "Zip không chứa 02_metadata/dataset_manifest.csv"
        shutil.copytree(Path(hit[0]).parents[1], cfg.data_dir)
    print("Đã đưa Data Pack vào:", cfg.data_dir)
else:
    raise FileNotFoundError(f"Chưa có Data Pack tại {cfg.data_dir}")

## 3. Các nhóm kiểm tra
| Nhóm | Nội dung | Mức lỗi |
|---|---|---|
| G1 | Kiểm kê file theo cấu trúc README | File lõi thiếu → FAIL · nhãn mẫu thiếu → WARN |
| G2 | SHA-256, dung lượng, encoding, số dòng/cột, kỳ dữ liệu so với manifest | FAIL |
| G3 | Tên/thứ tự cột, kiểu dữ liệu, không rỗng theo data_dictionary | FAIL |
| G4 | Khóa chính, đủ lưới 12×5, khóa ngoại, không có năm 2020 | FAIL |
| G5 | Đơn vị VND_million, tiền tệ, năm tài chính, cùng phân ngành, cờ giả lập | FAIL |
| G6 | Trường trùng giữa income statement và cashflow phải bằng nhau | FAIL |
| G7 | Khóa template đầu ra khớp dữ liệu/câu hỏi | FAIL |
| G8 | Kết quả hợp nhất master_panel | FAIL |

Mỗi hàm nhận `data_dir` làm tham số để có thể chạy trên bản sao (kiểm thử âm ở mục 8).

In [ ]:
import csv, hashlib, time
import pandas as pd
pd.set_option("display.max_colwidth", 120)

def rd(data_dir, rel):
    path, _ = locate_pack_file(rel, data_dir)          # chịu được layout khác README (03_sample/, hậu tố _Case3)
    return pd.read_csv(path, encoding="utf-8-sig")

# ---------- G1. Kiểm kê file
def g1_inventory(log, data_dir):
    present = {}
    for files_, warn in [(CORE_FILES, False), (METADATA_FILES, False), (LABEL_FILES, True), (TEMPLATE_FILES, False)]:
        for name, rel in files_.items():
            path, how = locate_pack_file(rel, data_dir)
            ok = path is not None
            present[name] = ok
            if ok and how != "standard":                 # tìm thấy nhưng không đúng vị trí README -> WARN có ghi rõ
                log.check("G1_inventory", f"exists:{name}", rel, False, f"có, nhưng ở vị trí khác README ({how})", warn_only=True)
            else:
                log.check("G1_inventory", f"exists:{name}", rel, ok,
                          "có" if ok else ("THIẾU – các bước dùng file này sẽ hoãn" if warn else "THIẾU – file bắt buộc"),
                          warn_only=warn)
    return present

# ---------- G2. Đối chiếu manifest
def g2_manifest(log, data_dir):
    man = rd(data_dir, METADATA_FILES["dataset_manifest"])
    for _, r in man.iterrows():
        rel, fname = r["relative_path"], r["file_name"]
        p, _how = locate_pack_file(rel, data_dir)
        if p is None:
            log.check("G2_manifest", f"present:{fname}", rel, False,
                      "Có trong manifest nhưng chưa được cung cấp", warn_only=rel.startswith("03_labels_sample"))
            continue
        raw = p.read_bytes()
        sha_ok = sha256_of(p) == r["sha256"]
        log.check("G2_manifest", f"sha256:{fname}", rel, sha_ok, f"sha256 {'khớp' if sha_ok else 'KHÔNG khớp'} manifest")
        log.check("G2_manifest", f"size:{fname}", rel, len(raw) == int(r["file_size_bytes"]),
                  f"{len(raw)} / {int(r['file_size_bytes'])} bytes")
        enc_ok = raw.startswith(b"\xef\xbb\xbf") if str(r["encoding"]).upper() == "UTF-8-SIG" else True
        log.check("G2_manifest", f"encoding:{fname}", rel, enc_ok, f"kỳ vọng {r['encoding']}")
        with open(p, encoding="utf-8-sig", newline="") as f:
            rows = list(csv.reader(f, delimiter=r["delimiter"]))
        n_rows, n_cols = len(rows) - 1, len(rows[0])
        log.check("G2_manifest", f"rows:{fname}", rel, n_rows == int(r["row_count"]), f"{n_rows} / {int(r['row_count'])} dòng")
        log.check("G2_manifest", f"cols:{fname}", rel, n_cols == int(r["column_count"]), f"{n_cols} / {int(r['column_count'])} cột")
        if pd.notna(r["period_field"]):
            df = rd(data_dir, rel)
            lo, hi = df[r["period_field"]].min(), df[r["period_field"]].max()
            log.check("G2_manifest", f"period:{fname}", rel, (lo, hi) == (int(r["period_min"]), int(r["period_max"])),
                      f"{lo}–{hi} / kỳ vọng {int(r['period_min'])}–{int(r['period_max'])}")
    listed = set(man["relative_path"])
    for name, rel in {**LABEL_FILES, **TEMPLATE_FILES}.items():
        p, _how = locate_pack_file(rel, data_dir)
        if p is not None and rel not in listed:
            log.check("G2_manifest", f"unlisted:{name}", rel, True,
                      f"Không có trong manifest; sha256={sha256_of(p)[:16]}… (ghi nhận để audit)")

# ---------- G3. Schema theo data_dictionary
def g3_schema(log, data_dir, tables):
    dd = rd(data_dir, METADATA_FILES["data_dictionary"])
    for fname, spec in dd.groupby("file_name", sort=False):
        name = fname.replace(".csv", "")
        if name not in tables:
            continue
        df, n_err = tables[name], 0
        expected = list(spec.sort_values("field_order")["field_name"])
        if list(df.columns) != expected:
            n_err += 1
            log.check("G3_schema", f"columns:{name}", fname, False,
                      f"thiếu={sorted(set(expected)-set(df.columns))} thừa={sorted(set(df.columns)-set(expected))}")
        for _, f in spec.iterrows():
            col = f["field_name"]
            if col not in df.columns:
                continue
            s = df[col]
            if f["nullable"] == "NO" and s.isna().any():
                n_err += 1; log.check("G3_schema", f"notnull:{name}.{col}", fname, False, f"{int(s.isna().sum())} giá trị rỗng")
            if f["data_type"] == "integer" and not (pd.api.types.is_integer_dtype(s) or (s.dropna() % 1 == 0).all()):
                n_err += 1; log.check("G3_schema", f"type:{name}.{col}", fname, False, "không phải số nguyên")
            if f["data_type"] == "decimal" and not pd.api.types.is_numeric_dtype(s):
                n_err += 1; log.check("G3_schema", f"type:{name}.{col}", fname, False, "không phải số")
        if n_err == 0:
            log.check("G3_schema", f"schema:{name}", fname, True, f"{len(expected)} trường đúng tên, thứ tự, kiểu, không rỗng")

# ---------- G4. Khóa & tính đầy đủ
FIN = ["income_statement", "balance_sheet", "cashflow", "financial_notes"]
def g4_keys(log, tables):
    ci = tables["company_info"]
    companies = sorted(ci["company_id"])
    log.check("G4_keys", "pk:company_info", "company_info.csv", ci["company_id"].is_unique, f"{len(companies)} company_id duy nhất")
    grid = {(c, y) for c in companies for y in YEARS}
    for name in FIN:
        df = tables[name]
        dup = int(df.duplicated(KEY).sum())
        log.check("G4_keys", f"pk:{name}", f"{name}.csv", dup == 0, f"{dup} khóa trùng")
        keys = set(map(tuple, df[KEY].values.tolist()))
        miss, extra = grid - keys, keys - grid
        log.check("G4_keys", f"grid:{name}", f"{name}.csv", not miss and not extra,
                  f"đủ {len(companies)}×{len(YEARS)}={len(grid)} company-year" if not miss and not extra
                  else f"thiếu={sorted(miss)[:5]} thừa={sorted(extra)[:5]}")
        bad = sorted(set(df["company_id"]) - set(companies))
        log.check("G4_keys", f"fk:{name}", f"{name}.csv", not bad, "mọi company_id có trong company_info" if not bad else f"lạ: {bad}")
    log.check("G4_keys", "no_2020", "01_data/*", all((tables[n]["fiscal_year"] >= 2021).all() for n in FIN),
              "Không có năm 2020 – chỉ tiêu cần năm trước để trống cho 2021")

# ---------- G5. Đơn vị & cờ giả lập
def g5_units(log, tables):
    ci = tables["company_info"]
    for col, val in [("statement_unit", "VND_million"), ("reporting_currency", "VND"),
                     ("fiscal_year_end", "12-31"), ("subindustry", "Packaged Food Manufacturing")]:
        vals = sorted(ci[col].astype(str).unique())
        log.check("G5_units", col, "company_info.csv", vals == [val], f"giá trị: {vals}")
    log.check("G5_units", "data_is_synthetic", "company_info.csv",
              ci["data_is_synthetic"].astype(str).str.upper().eq("TRUE").all(), "toàn bộ là dữ liệu giả lập")

# ---------- G6. Trường trùng giữa các báo cáo
def g6_overlap(log, tables):
    m = tables["income_statement"].merge(tables["cashflow"], on=KEY, suffixes=("_is", "_cf"))
    for col in ["net_income_vnd_mn", "depreciation_amortization_vnd_mn"]:
        diff = int((m[f"{col}_is"] != m[f"{col}_cf"]).sum())
        log.check("G6_overlap", f"same:{col}", "income_statement ↔ cashflow", diff == 0, f"{len(m)-diff}/{len(m)} dòng khớp")

# ---------- G7. Khớp template đầu ra
def g7_templates(log, tables):
    if "submission_output_template" in tables:
        t = tables["submission_output_template"]
        tk = set(map(tuple, t[KEY].values.tolist()))
        pk = set(map(tuple, tables["income_statement"][KEY].values.tolist()))
        log.check("G7_template", "keys:submission_output", "submission_output_template.csv", tk == pk, f"{len(tk)} khóa = {len(pk)} khóa dữ liệu")
        log.check("G7_template", "cols:submission_output", "submission_output_template.csv", t.shape[1] == 31, f"{t.shape[1]} cột")
    if {"qa_submission_template", "questions_sample"} <= tables.keys():
        a, b = set(tables["qa_submission_template"]["question_id"]), set(tables["questions_sample"]["question_id"])
        log.check("G7_template", "ids:qa_submission", "qa_submission_template.csv", a == b, f"{len(a)} question_id khớp questions_sample")
    if {"answer_key_sample", "questions_sample"} <= tables.keys():
        a, b = set(tables["answer_key_sample"]["question_id"]), set(tables["questions_sample"]["question_id"])
        log.check("G7_template", "ids:answer_key", "answer_key_sample.csv", a == b, f"{len(a)} câu có đáp án")

## 4. Hợp nhất `master_panel` (60 dòng × 68 cột) + `field_lineage`
Income statement làm gốc → ghép balance sheet, cashflow (bỏ 2 cột trùng đã kiểm ở G6), financial notes theo `company_id + fiscal_year` → ghép 3 cột hồ sơ từ company_info.

In [ ]:
def build_master(log, tables):
    ci_cols = ["company_id", "company_name", "ticker_simulated", "company_size_band"]
    base = tables["income_statement"].copy()
    lineage = [("income_statement.csv", c, "") for c in base.columns if c not in KEY]
    for name in ["balance_sheet", "cashflow", "financial_notes"]:
        df = tables[name]
        keep = [c for c in df.columns if c not in KEY and c not in base.columns]
        dup = [c for c in df.columns if c not in KEY and c in base.columns]
        base = base.merge(df[KEY + keep], on=KEY, how="inner", validate="one_to_one")
        lineage += [(f"{name}.csv", c, "") for c in keep]
        lineage += [(f"{name}.csv", c, "trùng – dùng bản income_statement") for c in dup]
    base = tables["company_info"][ci_cols].merge(base, on="company_id", how="right", validate="one_to_many")
    lineage += [("company_info.csv", c, "") for c in ci_cols[1:]]
    base = base.sort_values(KEY).reset_index(drop=True)
    log.check("G8_join", "row_count", "master_panel", len(base) == 60, f"{len(base)} dòng (kỳ vọng 60)")
    log.check("G8_join", "no_null", "master_panel", not base.isna().any().any(), f"{base.shape[1]} cột, không rỗng do join")
    lin = pd.DataFrame(lineage, columns=["source_file", "source_field", "note"])
    lin.insert(0, "master_column", lin["source_field"])
    return base, lin

def run_step0(data_dir):
    """Chạy toàn bộ Bước 0 trên một thư mục dữ liệu. Trả về (log, master, lineage, present)."""
    log = CheckLog()
    present = g1_inventory(log, data_dir)
    if not (all(present[k] for k in CORE_FILES) and present["dataset_manifest"] and present["data_dictionary"]):
        return log, None, None, present
    g2_manifest(log, data_dir)
    tables = {n: rd(data_dir, r) for n, r in {**CORE_FILES, **LABEL_FILES, **TEMPLATE_FILES}.items() if present.get(n)}
    g3_schema(log, data_dir, tables); g4_keys(log, tables); g5_units(log, tables)
    g6_overlap(log, tables); g7_templates(log, tables)
    master, lineage = build_master(log, tables)
    return log, master, lineage, present

## 5. Chạy Bước 0 & lưu kết quả lên Drive

In [ ]:
t0 = time.perf_counter()
RUN_ID = new_run_id("S0")
log, master, lineage, present = run_step0(cfg.data_dir)
ELAPSED_MS = (time.perf_counter() - t0) * 1000
core_ok = all(locate_pack_file(k)[0] is not None for k in CORE_FILES)
FINGERPRINT = (hashlib.sha256("".join(sha256_of(locate_pack_file(k)[0]) for k in sorted(CORE_FILES)).encode()).hexdigest()
               if core_ok else "N/A – thiếu file lõi")

checks, counts, VERDICT = log.df(), log.counts(), log.verdict()
if not core_ok:
    print("⚠ THIẾU FILE BẮT BUỘC – Bước 0 FAIL. Các file không tìm thấy:")
    display(log.issues())
    print("Kiểm tra cấu trúc thư mục: 01_data/, 02_metadata/ ... phải nằm NGAY trong", cfg.data_dir)
save_output(checks, "step0", "integrity_checks.csv")
outputs = ["integrity_checks.csv", "integrity_report.md"]
if not log.failed:
    save_output(master, "step0", "master_panel.csv")
    save_output(lineage, "step0", "field_lineage.csv")
    outputs = ["master_panel.csv", "field_lineage.csv"] + outputs

append_audit({"run_id": RUN_ID, "step": "step0_ingest", "verdict": VERDICT, "checks": counts,
              "data_fingerprint": FINGERPRINT, "files_missing": [k for k, v in present.items() if not v],
              "outputs": [f"03_outputs/step0/{o}" for o in outputs], "processing_time_ms": round(ELAPSED_MS, 1)})

print(f"Run ID     : {RUN_ID}")
print(f"Kết luận   : {VERDICT}  {counts}")
print(f"Fingerprint: {FINGERPRINT[:32]}")
print(f"Thời gian  : {ELAPSED_MS:.0f} ms")
print(f"Đã lưu vào : {cfg.out_dir / 'step0'}")

## 6. Xem kết quả kiểm tra

In [ ]:
display(log.summary())
issues = log.issues()
print("\nCác điểm cần lưu ý:" if len(issues) else "\nKhông có cảnh báo.")
if len(issues):
    display(issues)

In [ ]:
# Xem chi tiết một nhóm, đổi tên nhóm tùy ý: G1_inventory … G8_join
display(checks[checks["group"] == "G2_manifest"])

In [ ]:
if master is not None:
    print("master_panel:", master.shape)
    display(master.head(10))
    display(lineage.head(15))

## 7. Ghi báo cáo `integrity_report.md`

In [ ]:
desc = {"G1_inventory": "Kiểm kê file theo cấu trúc README",
        "G2_manifest": "SHA-256, dung lượng, encoding, số dòng/cột, kỳ dữ liệu so với manifest",
        "G3_schema": "Tên/thứ tự cột, kiểu dữ liệu, không rỗng theo data_dictionary",
        "G4_keys": "Khóa chính, đủ lưới 12×5, khóa ngoại, không có năm 2020",
        "G5_units": "Đơn vị VND_million, tiền tệ, năm tài chính, cùng phân ngành, cờ giả lập",
        "G6_overlap": "Trường trùng giữa 2 báo cáo phải bằng nhau",
        "G7_template": "Khóa template đầu ra khớp dữ liệu/câu hỏi",
        "G8_join": "Kết quả hợp nhất master_panel"}
L = ["# Báo cáo Bước 0 – Tiếp nhận & kiểm tra toàn vẹn dữ liệu", "",
     f"- **Run ID:** `{RUN_ID}`", f"- **Thời điểm:** {now_iso()}",
     f"- **Data fingerprint:** `{FINGERPRINT[:32]}`", f"- **Thời gian xử lý:** {ELAPSED_MS:.0f} ms",
     f"- **Kết luận:** **{VERDICT}** — {counts.get('PASS',0)} PASS · {counts.get('WARN',0)} WARN · {counts.get('FAIL',0)} FAIL", "",
     "| Nhóm | Nội dung | PASS | WARN | FAIL |", "|---|---|---:|---:|---:|"]
L += [f"| {g} | {desc.get(g,'')} | {r.PASS} | {r.WARN} | {r.FAIL} |" for g, r in log.summary().iterrows()]
L += ["", "## Các điểm cần lưu ý", ""]
L += (["| Trạng thái | Kiểm tra | Đối tượng | Chi tiết |", "|---|---|---|---|"] +
      [f"| {r.status} | {r.check_id} | `{r.target}` | {r.detail} |" for r in issues.itertuples()]) if len(issues) else ["Không có."]
L += ["", "## Quy ước dữ liệu thiếu", "",
      "Không có năm 2020: các chỉ tiêu cần năm trước để trống cho 2021, đánh dấu `INSUFFICIENT_HISTORY`, không tự sinh số liệu."]
report = cfg.step_dir("step0") / "integrity_report.md"
report.write_text("\n".join(L), encoding="utf-8")
print("Đã ghi:", report)

## 8. (Tùy chọn) Kiểm thử âm – chứng minh hệ thống phát hiện dữ liệu bị sửa
Sao chép Data Pack vào thư mục tạm, sửa **1 triệu đồng** doanh thu C006/2025 **trên bản sao**, chạy lại toàn bộ Bước 0 trên bản sao.
Kết quả mong đợi: `sha256:income_statement.csv` → **FAIL** và không xuất `master_panel`. Dữ liệu gốc trên Drive không bị chạm tới; thư mục tạm tự xóa khi xong.

In [ ]:
with tempfile.TemporaryDirectory() as tmp:
    tmp_pack = Path(tmp) / "pack"
    shutil.copytree(cfg.data_dir, tmp_pack)
    f = tmp_pack / CORE_FILES["income_statement"]
    f.write_bytes(f.read_bytes().replace(b"C006,2025,1299735", b"C006,2025,1299736"))   # sửa trên BẢN SAO
    neg_log, neg_master, _, _ = run_step0(tmp_pack)

print("Kết luận trên bản sao bị sửa:", neg_log.verdict(), neg_log.counts())
print("master_panel được xuất?", "KHÔNG (đúng như mong đợi)" if neg_log.failed else "CÓ")
display(neg_log.issues()[neg_log.issues()["status"] == "FAIL"])

## 9. Xem lịch sử chạy (audit log)

In [ ]:
display(read_audit().tail(10))